# Day 4 — Smoke-Test Training Pipeline
## AIT304 PCB Defect Detection · YOLO11n + Coordinate Attention

**Context:** This notebook is the Day 4 scaffold (handoff 4a). It mounts Drive,
syncs the repo and dataset, installs pinned dependencies, validates data integrity,
downloads pretrained weights, and snapshots the environment. No training is executed here.
The `model.train()` call is added in handoff 4b.

**Reference files:**
- [PROJECT_STATE.md](../PROJECT_STATE.md) — current phase and daily status
- [DECISIONS.md](../DECISIONS.md) — architectural reasoning log (see D-013, D-018, D-019)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, subprocess, sys

REPO_DRIVE = '/content/drive/MyDrive/pcb-defect-detection'
REPO_URL = 'https://github.com/pipiking123/pcb-defect-detection.git'

if not os.path.exists(REPO_DRIVE):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DRIVE], check=True)
else:
    # fail loudly on dirty repo
    r = subprocess.run(['git', '-C', REPO_DRIVE, 'status', '--porcelain'],
                       capture_output=True, text=True, check=True)
    assert r.stdout.strip() == '', f"Drive repo is dirty:\n{r.stdout}"
    subprocess.run(['git', '-C', REPO_DRIVE, 'pull', '--ff-only'], check=True)

os.chdir(REPO_DRIVE)
COMMIT_HASH = subprocess.run(['git', 'rev-parse', 'HEAD'],
                              capture_output=True, text=True, check=True).stdout.strip()
print(f"Repo at {REPO_DRIVE}, commit {COMMIT_HASH}")


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'ultralytics==8.3.40'], check=True)

import ultralytics
assert ultralytics.__version__ == '8.3.40', \
    f"Expected ultralytics 8.3.40, got {ultralytics.__version__}"
print(f"ultralytics {ultralytics.__version__} OK")


In [ ]:
import torch

print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Device 0: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
assert torch.cuda.is_available(), \
    "No CUDA. Switch Colab runtime to GPU (T4/L4/A100) and restart."


In [ ]:
import os, random, numpy as np, torch

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f"Seed set to {SEED}")


In [ ]:
gpu_name = torch.cuda.get_device_name(0)
if 'A100' in gpu_name:
    BATCH = 64; tier = 'A100'
elif 'L4' in gpu_name:
    BATCH = 32; tier = 'L4'
elif 'T4' in gpu_name:
    BATCH = 16; tier = 'T4'
else:
    BATCH = 16; tier = 'unknown'
    print(f"WARNING: unrecognized GPU '{gpu_name}', defaulting batch={BATCH}")
print(f"GPU tier: {tier}, batch size: {BATCH}")
print("NOTE: smoke-only batch adaptivity; metrics not comparable across tiers.")


In [ ]:
# Cell 3 — Dataset acquisition (Drive folder → Drive zip → fail loud)
# Per D-020: handles Windows-zip backslash entries cross-platform.
import shutil, zipfile
from pathlib import Path

DRIVE_REPO   = '/content/drive/MyDrive/pcb-defect-detection'  # confirmed: args.yaml project path
LOCAL_ROOT   = Path('/content/datasets')
LOCAL_DATA   = LOCAL_ROOT / 'deeppcb'
DRIVE_FOLDER = Path(DRIVE_REPO) / 'datasets' / 'deeppcb'
DRIVE_ZIP    = Path(DRIVE_REPO) / 'datasets' / 'deeppcb.zip'

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

if LOCAL_DATA.exists() and any(LOCAL_DATA.iterdir()):
    print(f'[cell3] Local dataset already present at {LOCAL_DATA}, skipping acquisition.')
elif DRIVE_FOLDER.exists() and any(DRIVE_FOLDER.iterdir()):
    print(f'[cell3] Path A: copying Drive folder {DRIVE_FOLDER} -> {LOCAL_DATA}')
    shutil.copytree(DRIVE_FOLDER, LOCAL_DATA, dirs_exist_ok=True)
elif DRIVE_ZIP.exists():
    print(f'[cell3] Path B: extracting Drive zip {DRIVE_ZIP} -> {LOCAL_ROOT}')
    with zipfile.ZipFile(DRIVE_ZIP) as zf:
        for entry in zf.infolist():
            # D-020: Windows Compress-Archive uses '\' in entry names; normalize.
            entry.filename = entry.filename.replace('\\', '/')
            zf.extract(entry, LOCAL_ROOT)
    assert LOCAL_DATA.exists(), (
        f'[cell3] Extraction completed but {LOCAL_DATA} missing — '
        f'check zip top-level structure.'
    )
else:
    raise FileNotFoundError(
        f'[cell3] Dataset not found. Expected either:\n'
        f'  - Drive folder: {DRIVE_FOLDER}\n'
        f'  - Drive zip:    {DRIVE_ZIP}\n'
        f'Run Day 3 converter and stage to Drive before re-running.'
    )

# Structural verification (fast fail before training)
for split in ('train', 'val', 'test'):
    for sub in ('images', 'labels'):
        p = LOCAL_DATA / sub / split
        assert p.exists(), f'[cell3] Missing required path: {p}'
        n = sum(1 for _ in p.iterdir())
        print(f'  {sub}/{split}: {n} files')

In [ ]:
YAML_PATH = '/content/datasets/deeppcb/deeppcb_colab.yaml'
yaml_content = """path: /content/datasets/deeppcb
train: images/train
val: images/val
test: images/test
nc: 6
names:
  0: open
  1: short
  2: mousebite
  3: spur
  4: copper
  5: pin-hole
"""
with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)
print(f"Wrote {YAML_PATH}")
print(yaml_content)


In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/datasets/deeppcb')
# Plate-aware split (D-024, a02964b): 83/20/72 plates -> 801/199/500 imgs.
# Not a clean 80/20 by image count because plates group unequally.
expected = {'train': 801, 'val': 199, 'test': 500}
counts = {}
for split, n in expected.items():
    imgs = sorted((DATA_ROOT / 'images' / split).glob('*.jpg'))
    lbls = sorted((DATA_ROOT / 'labels' / split).glob('*.txt'))
    counts[split] = (len(imgs), len(lbls))
    assert len(imgs) == n, f"{split}: expected {n} images, got {len(imgs)}"
    assert len(lbls) == n, f"{split}: expected {n} labels, got {len(lbls)}"
    img_stems = {p.stem for p in imgs}
    lbl_stems = {p.stem for p in lbls}
    assert img_stems == lbl_stems, \
        f"{split}: image/label stem mismatch ({len(img_stems ^ lbl_stems)} diff)"
    for lbl in lbls:
        text = lbl.read_text().strip()
        assert text, f"Empty label file: {lbl}"
        for line in text.splitlines():
            parts = line.split()
            assert len(parts) == 5, \
                f"Bad YOLO line ({len(parts)} fields, expected 5) in {lbl}: {line!r}"
            cid = int(parts[0])
            assert 0 <= cid <= 5, f"Bad class id {cid} in {lbl}"
            cx, cy, w, h = map(float, parts[1:])
            assert 0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0, \
                f"Bad center ({cx:.4f}, {cy:.4f}) in {lbl}: must be in [0,1]"
            assert 0.0 <= w <= 1.0 and 0.0 <= h <= 1.0, \
                f"Bad size ({w:.4f}, {h:.4f}) in {lbl}: must be in [0,1]"
            assert w > 0 and h > 0, f"Zero-area bbox in {lbl}: {line!r}"

# D-024 plate-disjoint invariant -- hard stop if trainval leaks
train_plates = {p.stem[:7] for p in (DATA_ROOT / 'images' / 'train').glob('*.jpg')}
val_plates   = {p.stem[:7] for p in (DATA_ROOT / 'images' / 'val').glob('*.jpg')}
test_plates  = {p.stem[:7] for p in (DATA_ROOT / 'images' / 'test').glob('*.jpg')}
print(f"plates -- train:{len(train_plates)} val:{len(val_plates)} test:{len(test_plates)}")
assert len(train_plates) == 83, f"expected 83 train plates, got {len(train_plates)}"
assert len(val_plates)   == 20, f"expected 20 val plates, got {len(val_plates)}"
assert train_plates.isdisjoint(val_plates), \
    f"D-024 VIOLATION: train∩val = {train_plates & val_plates}"
# Test overlap with trainval is documented upstream leakage (D-024 Option C); log, don't fail.
trainval_plates = train_plates | val_plates
upstream_leak = trainval_plates & test_plates
print(f"upstream test∩trainval leakage: {len(upstream_leak)} plates (D-024 Option C documented)")

print("Preflight data gate PASS:", counts)
TRAIN_N, VAL_N, TEST_N = counts['train'][0], counts['val'][0], counts['test'][0]

In [ ]:
from ultralytics import YOLO

WEIGHTS_CACHE = f"{REPO_DRIVE}/weights"
os.makedirs(WEIGHTS_CACHE, exist_ok=True)
os.chdir(WEIGHTS_CACHE)
try:
    model = YOLO('yolo11n.pt')
    print(f"Loaded yolo11n.pt: {sum(p.numel() for p in model.model.parameters())} params")
except Exception as e:
    raise RuntimeError(f"Failed to load yolo11n.pt: {e}")
os.chdir(REPO_DRIVE)


In [ ]:
import json, sys, datetime

ENV_PATH = f"{REPO_DRIVE}/runs/env_day4.json"
os.makedirs(os.path.dirname(ENV_PATH), exist_ok=True)
env = {
    'timestamp': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'ultralytics': ultralytics.__version__,
    'gpu_name': gpu_name,
    'gpu_memory_gb': round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
    'batch_size_selected': BATCH,
    'commit_hash': COMMIT_HASH,
    'seed': SEED,
    'data_yaml_path': YAML_PATH,
    'train_count': TRAIN_N,
    'val_count': VAL_N,
    'test_count': TEST_N,
    'pretrained_weights': 'yolo11n.pt',
    'notes': 'smoke test - batch-adaptive, metrics non-comparable across GPU tiers',
}
with open(ENV_PATH, 'w') as f:
    json.dump(env, f, indent=2)
print(json.dumps(env, indent=2))


> **Notebook hygiene reminder:** If committing from Colab UI rather than VS Code,
> run **Cell → All Output → Clear** before commit.
> The nbstripout filter only applies to VS Code/local commits.


In [ ]:
print("=" * 50)
print("READY TO TRAIN")
print("Handoff 4b will add the model.train() cell.")
print("=" * 50)


In [ ]:
from ultralytics import YOLO

# Pre-train guard: hard-stop if the target run dir already exists.
# Ultralytics 8.3.40 default behavior is to auto-increment to day4_vanilla2,
# which would silently drift away from the path our validation cell reads.
# An explicit assert keeps the run dir deterministic.
RUN_DIR = Path(f"{REPO_DRIVE}/runs/smoke/day4_vanilla")
assert not RUN_DIR.exists(), (
    f"Run dir {RUN_DIR} already exists. Delete or rename it before re-running, "
    f"or smoke results will drift to an incremented folder."
)

# Load pretrained YOLO11n from the Drive-cached weights (absolute path,
# per Day 4b handoff: avoid re-download from CWD).
model = YOLO(f"{WEIGHTS_CACHE}/yolo11n.pt")

# Locked smoke-test args. See DECISIONS.md D-013 (ultralytics pin),
# D-018 (augmentation policy: all off except fliplr=0.5).
try:
    results = model.train(
        data=YAML_PATH,
        epochs=5,
        imgsz=640,
        batch=BATCH,
        seed=SEED,
        workers=2,
        project=f"{REPO_DRIVE}/runs/smoke",
        name='day4_vanilla',
        save_period=-1,
        patience=100,
        plots=True,
        # D-018 augmentation policy — locked across Day 4 + Day 5+ ablation
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.0,
        degrees=0.0,
        translate=0.0,
        scale=0.0,
        shear=0.0,
        perspective=0.0,
        flipud=0.0,
        fliplr=0.5,
        mosaic=0.0,
    )
    print("Training completed.")
except Exception as e:
    raise RuntimeError(f"Training failed: {type(e).__name__}: {e}")

# Capture the authoritative run dir from the trainer that just executed.
# model.trainer.save_dir is the canonical owner in Ultralytics 8.3.40,
# not results.save_dir (which is a derived attribute).
TRAIN_RUN_DIR = Path(model.trainer.save_dir)
print(f"Train run dir: {TRAIN_RUN_DIR}")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Use TRAIN_RUN_DIR captured from model.trainer.save_dir in cell 14.
# This is the authoritative path in Ultralytics 8.3.40 and guards against
# stale-CSV false-PASS if a prior run left an old results.csv on disk.
RESULTS_CSV = TRAIN_RUN_DIR / 'results.csv'
assert RESULTS_CSV.exists(), f"results.csv not found at {RESULTS_CSV}"

df = pd.read_csv(RESULTS_CSV)
df.columns = [c.strip() for c in df.columns]  # ultralytics CSV can have stray whitespace
print(f"results.csv columns: {list(df.columns)}")
print(df)

# Locate metric columns defensively across ultralytics 8.3.x variants
def find_col(df, *candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of {candidates} found in columns: {list(df.columns)}")

epoch_col      = find_col(df, 'epoch')
box_loss_col   = find_col(df, 'train/box_loss')
cls_loss_col   = find_col(df, 'train/cls_loss')
dfl_loss_col   = find_col(df, 'train/dfl_loss')
map50_col      = find_col(df, 'metrics/mAP50(B)', 'metrics/mAP50')

# Gate 1: 5/5 epochs completed
gate1 = len(df) == 5
print(f"Gate 1 (5/5 epochs):           {'PASS' if gate1 else 'FAIL'} (rows={len(df)})")

# Gate 2: all numeric columns finite (no NaN, no Inf)
numeric = df.select_dtypes(include=[np.number])
all_finite = np.isfinite(numeric.to_numpy()).all()
gate2 = bool(all_finite)
print(f"Gate 2 (all losses finite):    {'PASS' if gate2 else 'FAIL'}")

# Gate 3: final val mAP50 > 0
final_map50 = float(df[map50_col].iloc[-1])
gate3 = final_map50 > 0.0
print(f"Gate 3 (final val mAP50 > 0):  {'PASS' if gate3 else 'FAIL'} (mAP50={final_map50:.4f})")

# Gate 4: train/box_loss strictly decreases epoch 1 -> epoch 5.
# box_loss is the primary detection regression signal; if it doesn't drop,
# the wiring is broken regardless of what cls/dfl loss do.
epoch1_box = float(df[box_loss_col].iloc[0])
epoch5_box = float(df[box_loss_col].iloc[-1])
gate4 = epoch5_box < epoch1_box
# Log cls/dfl losses for diagnostics but do not gate on them.
epoch1_cls = float(df[cls_loss_col].iloc[0])
epoch5_cls = float(df[cls_loss_col].iloc[-1])
epoch1_dfl = float(df[dfl_loss_col].iloc[0])
epoch5_dfl = float(df[dfl_loss_col].iloc[-1])
print(f"Gate 4 (box_loss decreases):   {'PASS' if gate4 else 'FAIL'} "
      f"(epoch1={epoch1_box:.4f} -> epoch5={epoch5_box:.4f})")
print(f"  diagnostic cls_loss: {epoch1_cls:.4f} -> {epoch5_cls:.4f}")
print(f"  diagnostic dfl_loss: {epoch1_dfl:.4f} -> {epoch5_dfl:.4f}")

all_pass = gate1 and gate2 and gate3 and gate4
print("=" * 50)
print(f"SMOKE TEST: {'PASS' if all_pass else 'FAIL'}")
print("=" * 50)

# Record summary for the post-execution PROJECT_STATE.md update
SMOKE_SUMMARY = {
    'run_dir': str(TRAIN_RUN_DIR),
    'epochs_completed': int(len(df)),
    'all_finite': bool(all_finite),
    'final_mAP50': final_map50,
    'epoch1_box_loss': epoch1_box,
    'epoch5_box_loss': epoch5_box,
    'epoch1_cls_loss': epoch1_cls,
    'epoch5_cls_loss': epoch5_cls,
    'epoch1_dfl_loss': epoch1_dfl,
    'epoch5_dfl_loss': epoch5_dfl,
    'overall': 'PASS' if all_pass else 'FAIL',
}
print(SMOKE_SUMMARY)

# Hard-assert overall result so a FAIL cannot silently commit.
assert all_pass, "SMOKE TEST FAILED -- see gate output above; do not commit results."